In [1]:
import pandas as pd     # REQUIRED
import seaborn as sns
import numpy as np
import matplotlib.pyplot as mpl
data = pd.read_csv("./winning_deck_results.csv", index_col=0) # REQUIRED
# Turn the won column into boolean
data['won'] = data['won'].map({' True': True, ' False': False}) # REQYIRED
data.head()

,num_moves,won,x0,x1,x2,x3,x4,x5,x6,x7,...,x42,x43,x44,x45,x46,x47,x48,x49,x50,x51
deck,,,,,,,,,,,,,,,,,,,,,
0,138,True,29,13,46,42,39,8,6,37,...,31,20,26,32,33,17,3,27,49,25
1,21,False,50,10,38,23,3,39,20,12,...,42,31,29,32,8,17,5,49,37,9
2,43,False,16,30,10,22,5,7,52,27,...,14,39,17,20,43,11,24,51,6,4
3,2,False,14,29,4,20,30,27,52,23,...,5,51,41,31,39,24,9,35,38,16
4,23,False,38,23,40,22,17,52,29,51,...,2,5,6,10,31,26,47,7,20,16


In [ ]:
# ENITRE BLOCK REQUIRED
visible_cards = data.iloc[:, [2, 9, 17, 20, 24, 27, 29]]
deck_cards = data.iloc[:, 30:]
known_cards = pd.concat([visible_cards, deck_cards], axis=1)
known_cards.columns = range(known_cards.shape[1])
known_cards.head()

In [ ]:
max_loss = data.loc[data['won'] == False, 'num_moves'].max()
min_win = data.loc[data['won'] == True, 'num_moves'].min()
print("Maximum moves to loss:", max_loss, " | Minimum moves to win:", min_win)

In [ ]:
wins = data['won'].value_counts()
mpl.pie(wins, labels=wins.index, autopct='%1.1f%%')
mpl.show()

In [3]:
known_cards = pd.concat([data.iloc[:, [2, 9, 17, 20, 24, 27, 29]], data.iloc[:, 30:]], axis=1)
known_cards.columns = range(known_cards.shape[1])
def remapping(data: pd.DataFrame, known: pd.DataFrame) -> pd.DataFrame:
    known = known.iloc[:, 2:].fillna(0)
    card_ids = data.values.astype(int)
    mask = card_ids > 0
    card_ids_adj = np.where(mask, card_ids - 1, 0)
    suits = card_ids_adj // 13
    ranks = (card_ids_adj % 13)
    colors = np.where(suits < 2, 0, 1)
    # handle unknowns
    ranks[~mask] = 0
    suits[~mask] = -1
    colors[~mask] = -1
    cols = []
    for i, col in enumerate(known.columns):
        cols.append(pd.DataFrame({
            f"{col}_rank": ranks[:, i],
            f"{col}_suit": suits[:, i],
            f"{col}_color": colors[:, i],
        }))
    remapped = pd.concat(cols, axis=1)
    return remapped
remapping(data, known_cards).head()

,2_rank,2_suit,2_color,3_rank,3_suit,3_color,4_rank,4_suit,4_color,5_rank,...,27_color,28_rank,28_suit,28_color,29_rank,29_suit,29_color,30_rank,30_suit,30_color
0,7,10,1,0,0,0,2,2,1,12,...,0,2,1,0,10,0,0,8,1,0
1,7,1,0,0,-1,-1,10,3,1,9,...,0,12,1,0,0,2,1,8,2,1
2,3,3,1,0,-1,-1,2,1,0,3,...,0,2,3,1,1,1,0,11,0,0
3,1,0,0,0,-1,-1,0,1,0,2,...,1,8,3,1,4,3,1,2,3,1
4,9,1,0,0,-1,-1,11,2,1,9,...,0,9,3,1,2,3,1,7,1,0


In [ ]:
# ENTIRE BLOCK REQUIRED
def count_valid_moves(ranks, colors):
    moves = 0
    n = len(ranks)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            if (
                ranks[i] > 0 and ranks[j] > 0 and
                colors[i] != colors[j] and
                ranks[i] == ranks[j] - 1
            ):
                moves += 1

    return moves
move_counts = []

for row in range(ranks.shape[0]):
    move_counts.append(count_valid_moves(ranks[row], colors[row]))

remapped["num_valid_moves"] = move_counts

In [ ]:
from sklearn.preprocessing import StandardScaler as StdS
from sklearn.linear_model import LogisticRegression as LReg
from sklearn.neural_network import MLPClassifier as MLP
from sklearn.neighbors import KNeighborsClassifier as KNN # REQUIRED
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier as GPC
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier as DTree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.model_selection import cross_val_score # REQUIRED
from alive_progress import alive_bar
from sklearn.preprocessing import LabelEncoder as le

In [ ]:
# ENTIRE BLOCK REQUIRED
X = remapped.values
y = data.iloc[:, 1].astype(int)
sc = StdS()
sc.fit(X)
X_sc = sc.transform(X)

In [ ]:
import numpy as np
print(np.mean(y))

# K Nearest Neighbors
Change `folds_to_test` and `neighbors_to_test` to modify the tests

In [ ]:
neighbors_to_test: int = 128
# K Nearest Neighbors Test
knn_results = pd.DataFrame(columns=['Neighbors', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(neighbors_to_test, force_tty=True) as bar:
    # Test different neighbors
    for i in range(3, neighbors_to_test+3):
        cvs = cross_val_score(KNN(i), X_sc, y, cv=10, n_jobs=-1) # REQUIRED
        mean: float = float(np.mean(cvs))
        std: float = float(np.std(cvs))
        knn_results.loc[len(knn_results)] = [i, mean, std]
        bar()
print(knn_results)
print(knn_results.describe())
print(knn_results.iloc[[int(knn_results['Accuracy'].idxmax())]])
sns.scatterplot(knn_results,x='Neighbors', y='Accuracy')

# Linear SVC
Add or remove from `tests_to_run` to change the number of tests

In [ ]:
tests_to_run = [.0001, .001, .01, .1, 1.0, 10.0]
# Linear SVC Test
lsvc_results = pd.DataFrame(columns=['C', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(len(tests_to_run), force_tty=True) as bar:
        # Test different neighbors
        for i in tests_to_run:
            cvs = cross_val_score(SVC(C=i, kernel="linear"), X_sc, y, cv=10, n_jobs=-1)
            mean: float = float(np.mean(cvs))
            std: float = float(np.std(cvs))
            lsvc_results.loc[len(lsvc_results)] = [i, mean, std]
            bar()
print(lsvc_results)
print(lsvc_results.describe())
print(lsvc_results.iloc[[int(lsvc_results['Accuracy'].idxmax())]])
sns.scatterplot(data=lsvc_results,x='C',y='Accuracy')

# SVC
Change `gamma_to_test` to change the number of tests

Uses `tests_to_run` from before

In [ ]:
gamma_to_test: int = 16
# SVC Test
svc_results = pd.DataFrame(columns=['C', 'Gamma', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(len(tests_to_run)*gamma_to_test, force_tty=True) as bar:
    for i in tests_to_run:
        for k in range(gamma_to_test):
            cvs = cross_val_score(SVC(C=i, gamma=gamma_to_test), X_sc, y, cv=10, n_jobs=-1)
            mean: float = float(np.mean(cvs))
            std: float = float(np.std(cvs))
            svc_results.loc[len(svc_results)] = [i, k, mean, std]
            bar()
print(svc_results)
print(svc_results.describe())
print(svc_results.iloc[[int(svc_results['Accuracy'].idxmax())]])
sns.scatterplot(data=svc_results,x='Gamma',hue='C',y='Accuracy')

In [ ]:
depth: int = 256
# Tree Test
dtree_results = pd.DataFrame(columns=['depth', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(depth, force_tty=True) as bar:
    for i in range(1, depth+1):
        cvs = cross_val_score(DTree(max_depth=i), X, y, cv=10, n_jobs=-1)
        mean: float = float(np.mean(cvs))
        std: float = float(np.std(cvs))
        dtree_results.loc[len(dtree_results)] = [i, mean, std]
        bar()
print(dtree_results)
print(dtree_results.describe())
print(dtree_results.iloc[[int(dtree_results['Accuracy'].idxmax())]])
sns.scatterplot(data=dtree_results,x='depth',y='Accuracy')

In [ ]:
estimators: int = 64
features: int = 8
# Tree Test
rforest_results = pd.DataFrame(columns=['estimators', 'features', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(estimators*features, force_tty=True) as bar:
    for o in range(1, estimators+1):
        for u in range(1, features+1):
            cvs = cross_val_score(RandomForestClassifier(max_depth=1, n_estimators=o, max_features=1), X_sc, y, cv=10, n_jobs=-1)
            mean: float = float(np.mean(cvs))
            std: float = float(np.std(cvs))
            rforest_results.loc[len(rforest_results)] = [o, u, mean, std]
            bar()
print(rforest_results)
print(rforest_results.describe())
print(rforest_results.iloc[[int(rforest_results['Accuracy'].idxmax())]])
sns.scatterplot(data=rforest_results,x='estimators',hue='features',y='Accuracy')
iterations: int = 32768
alpha: int = 16
# MLP Test
mlp_results = pd.DataFrame(columns=['Alpha', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(alpha, force_tty=True) as bar:
    for i in range(1, alpha+1):
        cvs = cross_val_score(MLP(alpha=o,max_iter=iterations), X_sc, y, cv=10, n_jobs=-1)
        mean: float = float(np.mean(cvs))
        std: float = float(np.std(cvs))
        mlp_results.loc[len(mlp_results)] = [i, mean, std]
        bar()
print(mlp_results)
print(mlp_results.describe())
print(mlp_results.iloc[[int(mlp_results['Accuracy'].idxmax())]])
sns.scatterplot(data=mlp_results,x='Alpha',y='Accuracy')

In [ ]:
# Ada Test
cvs = cross_val_score(AdaBoostClassifier(), X_sc, y, cv=10, n_jobs=-1)
mean: float = float(np.mean(cvs))
std: float = float(np.std(cvs))
print("Mean:", str(mean), "| Std:", str(std))

In [ ]:
# Gaussian Test
cvs = cross_val_score(GaussianNB(), X_sc, y, cv=10, n_jobs=-1)
mean: float = float(np.mean(cvs))
std: float = float(np.std(cvs))
print("Mean:", str(mean), "| Std:", std)

In [ ]:
state: int = 512
solver = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
# LReg Test
lreg_results = pd.DataFrame(columns=['Solver', 'State', 'Accuracy', 'Standard Deviation'])
# Test different folds
with alive_bar(state*len(solver), force_tty=True) as bar:
    for o in solver:
        for i in range(1, state+1):
            cvs = cross_val_score(LReg(random_state=state, solver=o), X_sc, y, cv=10, n_jobs=-1)
            mean: float = float(np.mean(cvs))
            std: float = float(np.std(cvs))
            lreg_results.loc[len(lreg_results)] = [o, i, mean, std]
            bar()
print(lreg_results)
print(lreg_results.describe())
print(lreg_results.iloc[[int(lreg_results['Accuracy'].idxmax())]])
sns.scatterplot(data=lreg_results,x='State',y='Accuracy',hue='Solver')

In [ ]:
from sklearn.metrics import RocCurveDisplay, auc
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
X_train = X_sc
cv = StratifiedKFold(n_splits=5) # just to 5 fold
classifier = LogisticRegression(random_state=1, solver='lbfgs')
tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)
fig, ax = plt.subplots()

# create and add ROC for each  fold
for i, (train, test) in enumerate(cv.split(X_train, y)): # iterator
    classifier.fit(X_train[train], y[train])
    viz = RocCurveDisplay.from_estimator(classifier, X_train[test], y[test],
                         name=f'ROC fold {i}',
                         alpha=0.3, lw=1, ax=ax)
    interp_tpr = np.interp(mean_fpr, viz.fpr, viz.tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(viz.roc_auc)

# add curve for random guessing
ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',
        label='Random guessing', alpha=.8)

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)

# add curve for mean scores
ax.plot(mean_fpr, mean_tpr, color='b',
        label=r'Mean ROC (AUC = %0.2f $\pm$ %0.2f)' % (mean_auc, std_auc),
        lw=2, alpha=.8)

# add curve for a perfect score
ax.plot([0, 0, 1],
        [0, 1, 1], linestyle=':', color='black', label='Perfect performance')

std_tpr = np.std(tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
ax.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2,
                label=r'$\pm$ 1 std. dev.')

ax.set(xlim=[-0.05, 1.05], ylim=[-0.05, 1.05],
       title="ROC Curve")
ax.legend(loc="lower right")
plt.show()

In [ ]:
winlossmap = {True: 'winning', False: 'losing'}
print(f'{round(confidence*100,2)}% chance of {winlossmap[prediction]}')

def main() -> None:
    remapping(data, known_cards)

if __name__ == "__main__":
    main()